# Explore raw Amazon Reviews 2023 data

Quick look at the raw `.jsonl.gz` review/meta files before writing any cleaning logic, so we know what we're actually dealing with in the 2023 dump (schema, join key, missing values, category names).

Dataset home: https://amazon-reviews-2023.github.io/ — key differences from the 2014/2018 versions we used for Beauty before:
- Files are `.jsonl.gz` (one strict JSON object per line), not the loose single-quoted `.json.gz` the 2014 dump used — no regex quote-fixing needed.
- Review join key is `parent_asin`, not `asin` (`asin` is now a specific variant; `parent_asin` is what item metadata is keyed by).
- Review fields: `user_id`, `parent_asin`, `asin`, `rating`, `timestamp` (**milliseconds**, not seconds), `title`, `text`, `verified_purchase`, `helpful_vote`.
- Meta fields: `parent_asin`, `title`, `main_category`, `average_rating`, `rating_number`, `features`, `description` (list of strings, not one string), `price`, `store`, `categories` (already a flat list — no nested `eval()` needed), `details` (dict, may contain `"Brand"`).

# 0. Import

In [ ]:
import gzip
import json
import itertools
import pandas as pd
from local_package.config.data import AMAZON_RAW_DIR

# 1. Configuration

In [2]:
# Path
CATEGORY = "All_Beauty"  # matches the filenames below; swap for any category from the dataset table

DATA_DIR = AMAZON_RAW_DIR / CATEGORY

REVIEW_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"

In [3]:
# Records peeking
N_PEEK = 3  # how many raw records to pretty-print
N_SAMPLE = 1000  # rows to load into a DataFrame; None = load everything (fine up to a few million rows)

In [4]:
def iter_jsonl_gz(path):
    """Yield one parsed record per line — 2023 dumps are strict JSON, so plain json.loads works."""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

In [5]:
def load_jsonl_gz(path, n_rows=None):
    records = itertools.islice(iter_jsonl_gz(path), n_rows) if n_rows else iter_jsonl_gz(path)
    return pd.DataFrame.from_records(records)

## Raw record peek

In [21]:
print(f"--- first {N_PEEK} review records ---")
for rec in itertools.islice(iter_jsonl_gz(REVIEW_FILE), N_PEEK):
    print(json.dumps(rec, indent=3, ensure_ascii=False)[:800], "\n")

--- first 3 review records ---
{
   "rating": 5.0,
   "title": "Such a lovely scent but not overpowering.",
   "text": "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
   "images": [],
   "asin": "B00YQ6X8EO",
   "parent_asin": "B00YQ6X8EO",
   "user_id": "AGKHLEW2SOWHNMFQIJGBECAF7INQ",
   "timestamp": 1588687728923,
   "helpful_vote": 0,
   "verified_purchase": true
} 

{
   "rating": 4.0,
   "title": "Works great but smells a little weird.",
   "text": "This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange coffee is offputting. (granted, I did know the smell was described but I was hoping it would be light)",
   "images": [],
   "asin": "B081TJ8YS3",
   "p

In [22]:
print(f"--- first {N_PEEK} meta records ---")
for rec in itertools.islice(iter_jsonl_gz(META_FILE), N_PEEK):
    print(json.dumps(rec, indent=2, ensure_ascii=False)[:800], "\n")

--- first 3 meta records ---
{
  "main_category": "All Beauty",
  "title": "Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)",
  "average_rating": 4.8,
  "rating_number": 10,
  "features": [],
  "description": [],
  "price": null,
  "images": [
    {
      "thumb": "https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg",
      "large": "https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg",
      "variant": "MAIN",
      "hi_res": null
    },
    {
      "thumb": "https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg",
      "large": "https://m.media-amazon.com/images/I/41w2yznfuZL.jpg",
      "variant": "PT01",
      "hi_res": "https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg"
    }
  ],
  "videos": [],
  "store": "Howard Products",
  "categories": [],
  "details": {
    "Package Dimensions":  

{
  "main_category": "All Beauty",
  "title": "Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba L

## Load into DataFrames

In [23]:
review_df = load_jsonl_gz(REVIEW_FILE, N_SAMPLE)
meta_df = load_jsonl_gz(META_FILE, N_SAMPLE)
print("reviews:", review_df.shape)
print("meta:", meta_df.shape)
review_df.head(3)

reviews: (1000, 10)
meta: (1000, 14)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True


In [24]:
meta_df.head(3)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,None
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,None
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,None


## Review stats

In [25]:
print("missing values per column:")
print(review_df.isna().mean().sort_values(ascending=False))

missing values per column:
rating               0.0
title                0.0
text                 0.0
images               0.0
asin                 0.0
parent_asin          0.0
user_id              0.0
timestamp            0.0
helpful_vote         0.0
verified_purchase    0.0
dtype: float64


In [26]:
print("rating distribution:")
print(review_df["rating"].value_counts().sort_index())

rating distribution:
rating
1.0     63
2.0     63
3.0    111
4.0    189
5.0    574
Name: count, dtype: int64


In [27]:
# timestamps are millisecond precision in the 2023 dump
ts = pd.to_datetime(review_df["timestamp"], unit="ms")
print("review time range:", ts.min(), "to", ts.max())

review time range: 2005-08-09 12:12:58 to 2023-03-11 21:45:21.367000


In [28]:
dup_pairs = review_df.duplicated(subset=["user_id", "parent_asin"]).sum()
print(f"duplicated (user_id, parent_asin) rows: {dup_pairs} / {len(review_df)}")

duplicated (user_id, parent_asin) rows: 4 / 1000


In [29]:
inter_per_user = review_df.groupby("user_id").size()
inter_per_item = review_df.groupby("parent_asin").size()
print("interactions per user:\n", inter_per_user.describe())
print("\ninteractions per item:\n", inter_per_item.describe())

interactions per user:
 count    415.000000
mean       2.409639
std        6.325388
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       81.000000
dtype: float64

interactions per item:
 count    921.000000
mean       1.085776
std        0.326749
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        4.000000
dtype: float64


## Meta stats

In [30]:
print("missing values per column:")
print(meta_df.isna().mean().sort_values(ascending=False))

missing values per column:
bought_together    1.000
price              0.799
store              0.099
average_rating     0.000
main_category      0.000
title              0.000
description        0.000
features           0.000
rating_number      0.000
images             0.000
videos             0.000
categories         0.000
details            0.000
parent_asin        0.000
dtype: float64


In [31]:
print("price stats (non-null):")
print(meta_df["price"].describe())
print(f"missing price: {meta_df['price'].isna().mean():.1%}")

price stats (non-null):
count    201.000000
mean      23.926517
std       25.104041
min        0.590000
25%        8.990000
50%       16.020000
75%       25.000000
max      179.950000
Name: price, dtype: float64
missing price: 79.9%


In [32]:
print("main_category counts:")
print(meta_df["main_category"].value_counts().head(20))

main_category counts:
main_category
All Beauty        993
Premium Beauty      7
Name: count, dtype: int64


In [33]:
flat_categories = meta_df["categories"].explode()
print("top leaf categories:")
print(flat_categories.value_counts().head(20))

top leaf categories:
Series([], Name: count, dtype: int64)


In [34]:
has_description = meta_df["description"].apply(lambda x: isinstance(x, list) and len(x) > 0)
print(f"items with non-empty description: {has_description.mean():.1%}")

items with non-empty description: 20.3%


## Review ↔ meta join coverage

In [35]:
coverage = review_df["parent_asin"].isin(meta_df["parent_asin"]).mean()
print(f"reviews whose parent_asin has meta: {coverage:.1%}")

reviews whose parent_asin has meta: 1.6%
